In [32]:
import pandas as pd
import html
from typing import List, Dict, Union, Optional
from pathlib import Path

def excel_to_bootstrap_html(
    path: str,
    sheet: Union[str, int, None] = 0,
    *,
    widths: Optional[Union[List[Union[str,float,int]], Dict[str, Union[str,float,int]]]] = None,
    # widths: Liste/Dikt pro Spalte; Einträge dürfen CSS-Strings ("120px","14ch","20%") oder Zahlen sein
    width_unit: str = "px",         # Einheit für numerische widths (z.B. "px", "ch", "em", "%")
    caption: Optional[str] = None,
    table_id: Optional[str] = None,
    embed_bootstrap: bool = True,
    classes: str = "table table-sm table-striped table-hover table-bordered align-middle",
    save_to: Optional[str] = None,
    container_align: str = "left",  # "left" | "center" | "right" (horizontale Position der Tabelle)
    max_table_width: Optional[str] = None,   # z.B. "1200px" oder "80ch" (optional)
) -> str:
    """
    Liest Excel/CSV und rendert eine Bootstrap-HTML-Tabelle.
    - Spaltenbreiten werden NICHT auf 100% normiert. Gib echte CSS-Einheiten ("px","ch","%") oder Zahlen (interpretiert als width_unit) an.
    - Tabelle wird als inline-table mit width:auto gerendert => füllt NICHT automatisch die gesamte Seitenbreite.
    - container_align steuert die horizontale Position (links/zentriert/rechts).
    """
    # 1) Daten laden
    if path.lower().endswith((".csv", ".tsv")):
        sep = "\t" if path.lower().endswith(".tsv") else ","
        df = pd.read_csv(path, sep=sep)
    else:
        df = pd.read_excel(path, sheet_name=sheet)

    if df.shape[1] == 0:
        raise ValueError("Die geladene Tabelle hat keine Spalten.")
    
    col_names = list(map(str, df.columns))
    n_cols = len(col_names)
    tid = table_id or f"tbl_{abs(hash(tuple(col_names)))%10**8}"

    # 2) Spaltenbreiten aufbereiten (KEIN Normalisieren)
    def _as_css(v):
        if isinstance(v, (int, float)):
            return f"{v}{width_unit}"
        v = str(v).strip()
        # wenn keine Einheit enthalten ist, fallback auf width_unit
        if any(v.endswith(u) for u in ("px","%","em","rem","ch","vw","vh")):
            return v
        return f"{v}{width_unit}"

    col_width_css: List[str] = []
    if widths is None:
        col_width_css = []  # kein colgroup -> natürliche Breite / Inhalte entscheiden
    elif isinstance(widths, dict):
        assigned = [widths.get(c, None) for c in col_names]
        col_width_css = [_as_css(w) if w is not None else None for w in assigned]
    else:
        wlist = list(widths)
        # auf n_cols matchen (ohne Normierung)
        if len(wlist) < n_cols:
            wlist = wlist + [None]*(n_cols - len(wlist))
        elif len(wlist) > n_cols:
            wlist = wlist[:n_cols]
        col_width_css = [_as_css(w) if w is not None else None for w in wlist]

    # 3) Head/CSS
    head_parts = []
    if embed_bootstrap:
        head_parts.append(
            '<link rel="stylesheet" '
            'href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.3/dist/css/bootstrap.min.css">'
        )

    align_css = {
        "left":   "text-align:left;",
        "center": "text-align:center;",
        "right":  "text-align:right;",
    }.get(container_align.lower(), "text-align:left;")

    table_extra_style = [
        "display:inline-table",   # verhindert 100%-Breite
        "width:auto",             # benutze Inhalt/colgroup
        "table-layout:auto",      # respektiert colgroup mit absoluten Breiten gut
        # Hinweis: bei Prozent-Spaltenbreiten wirken die Prozente relativ zur Tabellenbreite.
    ]
    if max_table_width:
        table_extra_style.append(f"max-width:{max_table_width}")
    table_style_str = ";".join(table_extra_style)

    head_parts.append(f"""
<style>
/* gleiche Zell-Ausrichtung überall sicherstellen */
#{tid} td, #{tid} th {{ text-align:left !important; }}
#{tid} thead th {{ position: sticky; top: 0; z-index: 2; background: var(--bs-body-bg, #fff); }}
#wrap_{tid} {{ {align_css} }}  /* horizontale Positionierung der Tabelle */
</style>
""".strip())

    # 4) Caption
    caption_html = f"<caption class='caption-top'>{html.escape(str(caption))}</caption>" if caption else ""

    # 5) colgroup (nur setzen, wenn Breiten angegeben)
    if col_width_css:
        colgroup = "<colgroup>" + "".join([
            f"<col style='width:{w};'>" if w is not None else "<col>"
            for w in col_width_css
        ]) + "</colgroup>"
    else:
        colgroup = ""  # keine festen Breiten -> natürliche Breite

    # 6) thead/tbody
    thead_html = "<thead><tr>" + "".join([f"<th scope='col'>{html.escape(str(c))}</th>" for c in col_names]) + "</tr></thead>"
    body_rows = []
    for _, row in df.iterrows():
        tds = []
        for c in col_names:
            val = "" if pd.isna(row[c]) else str(row[c])
            tds.append(f"<td>{html.escape(val)}</td>")
        body_rows.append("<tr>" + "".join(tds) + "</tr>")
    tbody_html = "<tbody>" + "".join(body_rows) + "</tbody>"

    # 7) Tabelle (ohne .table-responsive, um 100%-Breite zu vermeiden)
    table_html = (
        f"<div id='wrap_{tid}'>"
        f"<table id='{tid}' class='{classes}' style='{table_style_str}'>"
        f"{caption_html}{colgroup}{thead_html}{tbody_html}"
        f"</table></div>"
    )

    html_out = "\n".join(head_parts + [table_html])

    if save_to:
        p = Path(save_to)
        p.parent.mkdir(parents=True, exist_ok=True)
        with open(p, "w", encoding="utf-8") as f:
            f.write(html_out)

    return html_out

from IPython.display import HTML, display

html_str = excel_to_bootstrap_html(
    r"K:\Team\Böhmer_Michael\PAPER\Ergebnisse\Abbildungen_Tabellen\verteilung.xlsx",
    sheet="Tabelle1",
    widths=["40ch","20ch","20ch","20ch","20ch", "20ch"],  # echte CSS-Einheiten pro Spalte
    caption="Table 1. External performance",
    container_align="left",        # links positionieren
    max_table_width=None,          # optional: z.B. "1200px"
    save_to=r"K:\Team\Böhmer_Michael\PAPER\Ergebnisse\Abbildungen_Tabellen\verteilung.html"
)
display(HTML(html_str))


Feature,Motum Shapiro-p,Maestroni Shapiro-p,ΔMean (Motum−Maestroni),Cohen’s d,KS-p
inv cmj uni av. propulsive force,0.779,0.262,-6553.0,-5594.0,1.85e-40
uninv cmj uni av. propulsive force,0.436,0.106,-6615.0,-5255.0,1.85e-40
uninv cmj uni peak braking force,0.76,0.161,-6451.0,-3230.0,5.42e-31
inv cmj uni peak braking force,0.151,0.00897,-6611.0,-3201.0,4.82e-33
uninv cmj uni peak landing force,7.45e-05,0.256,1944.0,2252.0,9.76e-25
inv cmj uni av. braking force,0.0528,7.72e-10,-1952.0,-1803.0,8.57e-22
inv cmj uni peak landing force,0.00563,0.000169,1428.0,1797.0,2.32e-17
uninv cmj uni av. braking force,0.436,1.23e-09,-1664.0,-1571.0,5.41e-20
uninv cmj uni av. braking power,0.00764,0.766,2745.0,1544.0,3.82e-13
inv cmj uni av. braking power,0.0255,0.338,2454.0,1470.0,2.02e-14


In [17]:
import pandas as pd
import html
from typing import List, Dict, Union, Optional
from pathlib import Path
from IPython.display import HTML, display

def excel_to_bootstrap_html(
    path: str,
    sheet: Union[str, int, None] = 0,
    *,
    widths: Optional[Union[List[Union[str,float,int]], Dict[str, Union[str,float,int]]]] = None,
    width_unit: str = "px",
    caption: Optional[str] = None,
    table_id: Optional[str] = None,
    embed_bootstrap: bool = True,
    classes: str = "table table-sm table-striped table-hover table-bordered align-middle",
    save_to: Optional[str] = None,
    save_standalone: bool = True,      # ← NEU: beim Speichern vollständiges HTML-Dokument erzeugen
    container_align: str = "left",
    max_table_width: Optional[str] = None,
) -> str:
    # ---------------- Daten laden ----------------
    if path.lower().endswith((".csv", ".tsv")):
        sep = "\t" if path.lower().endswith(".tsv") else ","
        df = pd.read_csv(path, sep=sep)
    else:
        df = pd.read_excel(path, sheet_name=sheet)

    if df.shape[1] == 0:
        raise ValueError("Die geladene Tabelle hat keine Spalten.")

    col_names = list(map(str, df.columns))
    n_cols = len(col_names)
    tid = table_id or f"tbl_{abs(hash(tuple(col_names)))%10**8}"

    # --------------- Breiten (keine Normierung) ---------------
    def _as_css(v):
        if v is None:
            return None
        if isinstance(v, (int, float)):
            return f"{v}{width_unit}"
        v = str(v).strip()
        if any(v.endswith(u) for u in ("px","%","em","rem","ch","vw","vh")):
            return v
        return f"{v}{width_unit}"

    if widths is None:
        col_width_css: List[Optional[str]] = [None]*n_cols
    elif isinstance(widths, dict):
        assigned = [widths.get(c, None) for c in col_names]
        col_width_css = [_as_css(w) for w in assigned]
    else:
        wlist = list(widths)
        if len(wlist) < n_cols:
            wlist += [None]*(n_cols-len(wlist))
        elif len(wlist) > n_cols:
            wlist = wlist[:n_cols]
        col_width_css = [_as_css(w) for w in wlist]

    # ----------------- Head/CSS -------------------
    head_parts = []
    if embed_bootstrap:
        head_parts.append(
            '<link rel="stylesheet" '
            'href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.3/dist/css/bootstrap.min.css">'
        )

    align_css = {
        "left":   "text-align:left;",
        "center": "text-align:center;",
        "right":  "text-align:right;",
    }.get(container_align.lower(), "text-align:left;")

    table_extra_style = [
        "display:inline-table",
        "width:auto",
        "table-layout:auto",
    ]
    if max_table_width:
        table_extra_style.append(f"max-width:{max_table_width}")
    table_style_str = ";".join(table_extra_style)

    head_parts.append(f"""
<style>
#{tid} td, #{tid} th {{ text-align:left !important; }}
#{tid} thead th {{ position: sticky; top: 0; z-index: 2; background: var(--bs-body-bg, #fff); }}
#wrap_{tid} {{ {align_css} }}
</style>
""".strip())

    head_html = "\n".join(head_parts)

    # ---------------- Table HTML ------------------
    caption_html = f"<caption class='caption-top'>{html.escape(str(caption))}</caption>" if caption else ""
    colgroup = (
        "<colgroup>" + "".join(
            [f"<col style='width:{w};'>" if w else "<col>" for w in col_width_css]
        ) + "</colgroup>"
    ) if col_width_css else ""

    thead_html = "<thead><tr>" + "".join(
        [f"<th scope='col'>{html.escape(str(c))}</th>" for c in col_names]
    ) + "</tr></thead>"

    body_rows = []
    for _, row in df.iterrows():
        tds = [f"<td>{html.escape('' if pd.isna(row[c]) else str(row[c]))}</td>" for c in col_names]
        body_rows.append("<tr>" + "".join(tds) + "</tr>")
    tbody_html = "<tbody>" + "".join(body_rows) + "</tbody>"

    table_only_html = (
        f"<div id='wrap_{tid}'>"
        f"<table id='{tid}' class='{classes}' style='{table_style_str}'>"
        f"{caption_html}{colgroup}{thead_html}{tbody_html}"
        f"</table></div>"
    )

    # Für Notebook-Display: Head + Snippet (wie zuvor)
    html_out = "\n".join([head_html, table_only_html])

    # --------------- Speichern (robuster) ---------------
    if save_to:
        p = Path(save_to)
        p.parent.mkdir(parents=True, exist_ok=True)
        if save_standalone:
            # Vollständiges, eigenständiges HTML-Dokument
            doc = f"""<!doctype html>
<html lang="de">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
{head_html}
<title>Table Export</title>
</head>
<body>
{table_only_html}
</body>
</html>"""
            with open(p, "w", encoding="utf-8") as f:
                f.write(doc)
        else:
            # Nur das bisherige Fragment (wie vorher)
            with open(p, "w", encoding="utf-8") as f:
                f.write(html_out)

    return html_out

html_str = excel_to_bootstrap_html(
    r"K:\Team\Böhmer_Michael\PAPER\Datensätze_final\Datenübersicht\CMJ_ISO_Mean_SD_korrigiert - Kopie.xlsx",
    sheet="Tabelle1",
    widths=["20ch","14ch","10ch","30ch","18ch"],  # echte CSS-Breiten
    caption="Table 1. External performance",
    container_align="left",
    save_to=r"K:\Team\Böhmer_Michael\PAPER\Datensätze_final\Datenübersicht\tabelle.html",  
    save_standalone=True                               
)
display(HTML(html_str))

Features,Motum (Injured),Motum (Uninjured),Maestroni (Injured),Maestroni (Uninjured)
Sample size,n = 32,n = 34,n = 36,n = 35
Demographic features,,,,
Age (y),21.344 ± 4.382,25.794 ± 3.540,23.286 ± 4.144,23.856 ± 2.817
Mass (kg),73.329 ± 13.048,78.111 ± 11.923,71.181 ± 10.208,71.614 ± 6.264
Height (m),1.796 ± 0.052,1.786 ± 0.088,1.723 ± 6.942,1.738 ± 5.360
CMJ Braking features,,,,
INV_CMJ_uni_Peak braking force,9.985 ± 1.784,11.374 ± 1.648,17.113 ± 2.547,17.609 ± 1.889
UNINV_CMJ_uni_Peak braking force,11.299 ± 1.778,10.999 ± 1.847,18.007 ± 2.074,17.254 ± 2.192
INV_CMJ_uni_Av. braking force,7.394 ± 1.471,8.464 ± 1.356,10.050 ± 0.464,9.810 ± 0.070
UNINV_CMJ_uni_Av. braking force,8.405 ± 1.389,8.113 ± 1.539,10.051 ± 0.466,9.825 ± 0.079
